In [1]:
import os
import pandas as pd
from optbinning import BinningProcess
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.environ["DB_URL"])

df = pd.read_sql("SELECT * FROM loan_features", engine)
print(df.shape, df["default_flag"].sum())

(682, 44) 76


In [2]:
# Features to bin (continuous only; drop IDs, dates, target, categoricals)
features = [
    "age_at_loan", "account_age_months", "loan_amount", "duration", "payments",
    "avg_balance_3m", "avg_balance_6m", "avg_balance_12m",
    "sd_balance_3m", "sd_balance_6m",
    "min_balance_3m", "min_balance_6m",
    "pct_days_negative_6m", "balance_at_grant", "balance_change_90d",
    "monthly_inflow_3m", "monthly_inflow_6m",
    "monthly_outflow_3m", "monthly_outflow_6m",
    "n_credits_6m", "n_debits_6m", "cash_out_6m",
    "n_penalty_interest_12m", "has_pension_12m", "has_card_at_grant",
    "avg_salary", "unemployment_rate_95", "entrepreneurs_per_1000",
    "crimes_95", "urban_ratio", "inhabitants",
    "payment_to_inflow_6m", "loan_to_annual_salary",
    "inflow_outflow_ratio_6m", "cash_share_of_outflow_6m", "balance_to_loan"
]

X = df[features].copy()
y = df["default_flag"].copy()

# Run binning process
binning_process = BinningProcess(variable_names=features)
binning_process.fit(X, y)

# IV table
iv_table = binning_process.summary()
iv_table = iv_table.sort_values("iv", ascending=False).reset_index(drop=True)
print(iv_table[["name", "iv"]].to_string())

                        name        iv
0             min_balance_6m  2.036858
1             min_balance_3m  1.822079
2             avg_balance_3m  1.082498
3            balance_to_loan  1.079202
4           balance_at_grant  1.024285
5            avg_balance_12m  0.812421
6             avg_balance_6m  0.803742
7       payment_to_inflow_6m  0.710246
8   cash_share_of_outflow_6m  0.669569
9               n_credits_6m  0.561806
10                  payments  0.517638
11         monthly_inflow_3m  0.411108
12     loan_to_annual_salary  0.406009
13             sd_balance_6m  0.374347
14   inflow_outflow_ratio_6m  0.357097
15               loan_amount   0.35659
16        monthly_outflow_3m  0.351946
17        monthly_outflow_6m  0.329065
18               n_debits_6m  0.261974
19               cash_out_6m    0.2519
20         monthly_inflow_6m  0.230903
21        account_age_months  0.222553
22             sd_balance_3m  0.210515
23               inhabitants  0.196602
24                avg_sal

In [3]:
# Keep features with IV > 0.1
selected = iv_table[iv_table["iv"] > 0.1]["name"].tolist()
print(f"Selected {len(selected)} features")
print(selected)

Selected 28 features
['min_balance_6m', 'min_balance_3m', 'avg_balance_3m', 'balance_to_loan', 'balance_at_grant', 'avg_balance_12m', 'avg_balance_6m', 'payment_to_inflow_6m', 'cash_share_of_outflow_6m', 'n_credits_6m', 'payments', 'monthly_inflow_3m', 'loan_to_annual_salary', 'sd_balance_6m', 'inflow_outflow_ratio_6m', 'loan_amount', 'monthly_outflow_3m', 'monthly_outflow_6m', 'n_debits_6m', 'cash_out_6m', 'monthly_inflow_6m', 'account_age_months', 'sd_balance_3m', 'inhabitants', 'avg_salary', 'entrepreneurs_per_1000', 'balance_change_90d', 'crimes_95']


In [4]:
# Trim to a clean, non-redundant set of ~10 features
# Logic: keep the best IV per conceptual group, drop near-duplicates
scorecard_features = [
    "min_balance_6m",          # best balance floor signal (IV 2.04)
    "avg_balance_3m",          # level signal, less correlated with min than 6m avg (IV 1.08)
    "cash_share_of_outflow_6m",# spending behaviour (IV 0.67)
    "payment_to_inflow_6m",    # affordability ratio (IV 0.71)
    "n_credits_6m",            # income regularity (IV 0.56)
    "loan_to_annual_salary",   # loan size vs income (IV 0.41)
    "monthly_outflow_3m",      # spending level (IV 0.35)
    "account_age_months",      # relationship length (IV 0.22)
    "avg_salary",              # district wealth (IV 0.17)
    "balance_change_90d",      # trend signal (IV 0.12)
]

print(f"{len(scorecard_features)} features selected for scorecard")

10 features selected for scorecard


In [5]:
from optbinning import Scorecard
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Refit binning process on selected features only
bp = BinningProcess(variable_names=scorecard_features)

# Scorecard with logistic regression
scorecard = Scorecard(
    binning_process=bp,
    estimator=LogisticRegression(C=0.01, max_iter=1000, solver="lbfgs"),
    scaling_method="pdo_odds",
    scaling_method_params={"pdo": 20, "odds": 1, "scorecard_points": 600}
)

scorecard.fit(X[scorecard_features], y)
print("Scorecard fitted.")

Scorecard fitted.


In [6]:
# The scorecard table: points per bin for each feature
sc_table = scorecard.table(style="summary")
print(sc_table.to_string())

                   Variable                     Bin     Points
0            min_balance_6m          (-inf, 389.50)  33.447570
1            min_balance_6m       [389.50, 8036.60)  50.592738
2            min_balance_6m     [8036.60, 12604.30)  75.439215
3            min_balance_6m    [12604.30, 14792.25)  84.929929
4            min_balance_6m         [14792.25, inf)  89.399655
5            min_balance_6m                 Special  65.073301
6            min_balance_6m                 Missing  65.073301
0            avg_balance_3m        (-inf, 18907.93)  47.455468
1            avg_balance_3m    [18907.93, 23559.46)  58.082659
2            avg_balance_3m    [23559.46, 29807.53)  63.964391
3            avg_balance_3m    [29807.53, 34178.06)  65.336906
4            avg_balance_3m    [34178.06, 43330.58)  67.702392
5            avg_balance_3m    [43330.58, 53951.40)  72.316885
6            avg_balance_3m    [53951.40, 64747.96)  73.599772
7            avg_balance_3m         [64747.96, inf)  76

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Out-of-fold scores for honest AUC estimate
scores = scorecard.predict_proba(X[scorecard_features])[:, 1]
points = scorecard.score(X[scorecard_features])

print(f"Train ROC-AUC: {roc_auc_score(y, scores):.3f}")
print(f"Train PR-AUC:  {average_precision_score(y, scores):.3f}")
print(f"\nScore range: {points.min():.0f} – {points.max():.0f}")
print(f"Mean score:  {points.mean():.0f}")

Train ROC-AUC: 0.922
Train PR-AUC:  0.718

Score range: 578 – 718
Mean score:  671


In [8]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Cross-validated AUC on scorecard probabilities
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(y))

for train_idx, val_idx in cv.split(X[scorecard_features], y):
    sc_cv = Scorecard(
        binning_process=BinningProcess(variable_names=scorecard_features),
        estimator=LogisticRegression(C=0.01, max_iter=1000, solver="lbfgs"),
        scaling_method="pdo_odds",
        scaling_method_params={"pdo": 20, "odds": 1, "scorecard_points": 600}
    )
    sc_cv.fit(X[scorecard_features].iloc[train_idx], y.iloc[train_idx])
    oof_probs[val_idx] = sc_cv.predict_proba(X[scorecard_features].iloc[val_idx])[:, 1]

print(f"CV ROC-AUC: {roc_auc_score(y, oof_probs):.3f}")
print(f"CV PR-AUC:  {average_precision_score(y, oof_probs):.3f}")

CV ROC-AUC: 0.878
CV PR-AUC:  0.606


In [9]:
# Add scorecard output to df
df["sc_score"] = scorecard.score(X[scorecard_features])
df["sc_prob"]  = scorecard.predict_proba(X[scorecard_features])[:, 1]

# Default rate by score band (every 10 points)
df["score_band"] = pd.cut(df["sc_score"], bins=range(570, 730, 10))
summary = df.groupby("score_band", observed=True).agg(
    loans=("default_flag", "count"),
    defaults=("default_flag", "sum"),
    default_rate=("default_flag", "mean")
).reset_index()
summary["default_rate"] = (summary["default_rate"] * 100).round(1)
print(summary.to_string())

    score_band  loans  defaults  default_rate
0   (570, 580]      1         1         100.0
1   (580, 590]      5         5         100.0
2   (590, 600]      9         9         100.0
3   (600, 610]      8         8         100.0
4   (610, 620]     16         9          56.2
5   (620, 630]     33        16          48.5
6   (630, 640]     42        10          23.8
7   (640, 650]     41         7          17.1
8   (650, 660]     59         4           6.8
9   (660, 670]     49         1           2.0
10  (670, 680]     88         2           2.3
11  (680, 690]    132         4           3.0
12  (690, 700]    130         0           0.0
13  (700, 710]     55         0           0.0
14  (710, 720]     14         0           0.0


In [10]:
from sqlalchemy import text

# Write scorecard scores to Postgres
out_sc = df[["loan_id", "sc_score", "sc_prob", "default_flag"]].copy()
out_sc["sc_score"] = out_sc["sc_score"].round(0).astype(int)
out_sc["sc_prob"]  = out_sc["sc_prob"].round(4)

out_sc.to_sql("scorecard_predictions", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    conn.execute(text("ALTER TABLE scorecard_predictions ADD PRIMARY KEY (loan_id)"))
    conn.commit()

print(f"Written {len(out_sc)} rows to scorecard_predictions")
print(out_sc.head())

Written 682 rows to scorecard_predictions
   loan_id  sc_score  sc_prob  default_flag
0     4959       697   0.0339             0
1     4961       614   0.3779             1
2     4962       700   0.0307             0
3     4967       626   0.2901             1
4     4968       689   0.0434             0


In [12]:
# Two-stage model
# Stage 1: Scorecard declines below 620 (clear rejects)
# Stage 2: XGBoost on borderline cases (score 620-650)
# Get scorecard scores (already computed as df["sc_score"])
# Merge with existing df
df_2stage = df.copy()

# Stage 1 decisions
df_2stage["stage1_decision"] = "approve"
df_2stage.loc[df_2stage["sc_score"] < 620, "stage1_decision"] = "decline"

# Count stage 1 outcomes
print("Stage 1 (scorecard < 620):")
print(df_2stage.groupby("stage1_decision")["default_flag"].agg(["count","sum","mean"]).round(3))


Stage 1 (scorecard < 620):
                 count  sum   mean
stage1_decision                   
approve            643   44  0.068
decline             39   32  0.821


In [13]:
# Stage 2: XGBoost on borderline cases (620-650)
# Split approved into clear approvals and borderline
df_2stage["zone"] = "clear_approve"
df_2stage.loc[df_2stage["stage1_decision"] == "decline", "zone"] = "stage1_decline"
df_2stage.loc[(df_2stage["stage1_decision"] == "approve") & 
              (df_2stage["sc_score"] < 650), "zone"] = "borderline"

print("Zone breakdown:")
print(df_2stage.groupby("zone")["default_flag"].agg(["count","sum","mean"]).round(3))

Zone breakdown:
                count  sum   mean
zone                             
borderline        116   33  0.284
clear_approve     527   11  0.021
stage1_decline     39   32  0.821


In [15]:
# Drop string columns from features
exclude_cols = ["loan_id","account_id","loan_date","status","default_flag",
                "sc_score","sc_prob","score_band","stage1_decision","zone",
                "gender","statement_frequency","region"]

feature_cols = [c for c in df_2stage.columns if c not in exclude_cols]

# Borderline group only
border_mask = df_2stage["zone"] == "borderline"
X_border = df_2stage.loc[border_mask, feature_cols]
y_border = df_2stage.loc[border_mask, "default_flag"]

print(f"Borderline: {border_mask.sum()} loans, {y_border.sum()} defaults")
print(f"Features: {len(feature_cols)}")

# XGBoost OOF predictions on borderline group
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=8, random_state=42, eval_metric="aucpr")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_border = np.zeros(border_mask.sum())

for train_idx, val_idx in cv.split(X_border, y_border):
    xgb.fit(X_border.iloc[train_idx], y_border.iloc[train_idx])
    oof_border[val_idx] = xgb.predict_proba(X_border.iloc[val_idx])[:, 1]

df_2stage.loc[border_mask, "xgb_prob"] = oof_border

print(f"XGBoost OOF mean PD (borderline): {oof_border.mean():.3f}")
print(f"Observed default rate (borderline): {y_border.mean():.3f}")

Borderline: 116 loans, 33 defaults
Features: 37
XGBoost OOF mean PD (borderline): 0.339
Observed default rate (borderline): 0.284


In [16]:
# Apply cut-off of 0.25 to borderline XGBoost predictions
cutoff = 0.25

df_2stage["final_decision"] = "approve"
# Stage 1 declines
df_2stage.loc[df_2stage["zone"] == "stage1_decline", "final_decision"] = "decline"
# Stage 2 declines (borderline + XGBoost above cut-off)
df_2stage.loc[(df_2stage["zone"] == "borderline") & 
              (df_2stage["xgb_prob"] > cutoff), "final_decision"] = "decline"

# Results
total = len(df_2stage)
declined = (df_2stage["final_decision"] == "decline").sum()
defaults_caught = df_2stage.loc[df_2stage["final_decision"] == "decline", "default_flag"].sum()
good_declined = declined - defaults_caught

print("=== Two-Stage Model Results ===")
print(f"Total loans:        {total}")
print(f"Total declined:     {declined}")
print(f"Defaults caught:    {defaults_caught} / 76 ({defaults_caught/76*100:.1f}%)")
print(f"Good loans declined:{good_declined}")
print(f"\nCompare to single-stage logistic at 0.15 cut-off:")
print(f"Declined: 127 | Defaults caught: 55 (72%) | Good declined: 72")

=== Two-Stage Model Results ===
Total loans:        682
Total declined:     91
Defaults caught:    49 / 76 (64.5%)
Good loans declined:42

Compare to single-stage logistic at 0.15 cut-off:
Declined: 127 | Defaults caught: 55 (72%) | Good declined: 72


In [17]:
from optbinning import Scorecard, BinningProcess
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Full feature set for XGBoost (no strings)
xgb_features = [c for c in df.columns if c not in 
                ["loan_id","account_id","loan_date","status","default_flag",
                 "gender","statement_frequency","region",
                 "sc_score","sc_prob","score_band"]]

X_full = df[xgb_features].copy()
y_full = df["default_flag"].copy()
sc_X   = df[scorecard_features].copy()

oof_2stage = np.zeros(len(y_full))

for train_idx, val_idx in cv.split(X_full, y_full):
    # 1. Fit scorecard on train fold
    sc_cv = Scorecard(
        binning_process=BinningProcess(variable_names=scorecard_features),
        estimator=LogisticRegression(C=0.01, max_iter=1000, solver="lbfgs"),
        scaling_method="pdo_odds",
        scaling_method_params={"pdo": 20, "odds": 1, "scorecard_points": 600}
    )
    sc_cv.fit(sc_X.iloc[train_idx], y_full.iloc[train_idx])
    
    # 2. Get scores on val fold
    val_scores = sc_cv.score(sc_X.iloc[val_idx])
    val_probs  = sc_cv.predict_proba(sc_X.iloc[val_idx])[:, 1]
    
    # 3. Stage 1: decline < 620, clear approve > 650
    border_mask_val = (val_scores >= 620) & (val_scores < 650)
    
    # 4. XGBoost on borderline train samples
    train_scores = sc_cv.score(sc_X.iloc[train_idx])
    border_mask_train = (train_scores >= 620) & (train_scores < 650)
    
    X_b_train = X_full.iloc[train_idx][border_mask_train]
    y_b_train = y_full.iloc[train_idx][border_mask_train]
    
    # 5. Combine final probabilities
    final_probs = val_probs.copy()
    
    if border_mask_val.sum() > 0 and len(y_b_train) > 1 and y_b_train.sum() > 0:
        xgb_cv = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               scale_pos_weight=8, random_state=42, eval_metric="aucpr")
        xgb_cv.fit(X_b_train, y_b_train)
        border_idx = np.where(border_mask_val)[0]
        final_probs[border_idx] = xgb_cv.predict_proba(
            X_full.iloc[val_idx].iloc[border_idx])[:, 1]
    
    # Stage 1 declines get prob 1.0
    stage1_mask = val_scores < 620
    final_probs[stage1_mask] = 1.0
    
    oof_2stage[val_idx] = final_probs

roc_2s = roc_auc_score(y_full, oof_2stage)
pr_2s  = average_precision_score(y_full, oof_2stage)

print(f"Two-stage CV ROC-AUC: {roc_2s:.3f}")
print(f"Two-stage CV PR-AUC:  {pr_2s:.3f}")
print()
print("Full comparison (all CV):")
print(f"WoE Scorecard:       ROC 0.878  PR 0.606")
print(f"Logistic Regression: ROC 0.895  PR 0.658")
print(f"Random Forest:       ROC 0.910  PR 0.676")
print(f"XGBoost:             ROC 0.918  PR 0.679")
print(f"Two-stage:           ROC {roc_2s:.3f}  PR {pr_2s:.3f}")

Two-stage CV ROC-AUC: 0.823
Two-stage CV PR-AUC:  0.505

Full comparison (all CV):
WoE Scorecard:       ROC 0.878  PR 0.606
Logistic Regression: ROC 0.895  PR 0.658
Random Forest:       ROC 0.910  PR 0.676
XGBoost:             ROC 0.918  PR 0.679
Two-stage:           ROC 0.823  PR 0.505


In [18]:
from sqlalchemy import text

# Save two-stage decisions with loan_id to Postgres
two_stage_out = df_2stage[["loan_id", "final_decision", "default_flag"]].copy()
two_stage_out.to_sql("two_stage_decisions", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    conn.execute(text("ALTER TABLE two_stage_decisions ADD PRIMARY KEY (loan_id)"))
    conn.commit()

print(f"Written {len(two_stage_out)} rows")
print(two_stage_out["final_decision"].value_counts())

Written 682 rows
final_decision
approve    591
decline     91
Name: count, dtype: int64
